# Table of Contents

1. [Project Overview](#Project-Overview)
2. [Dataset](#dataset)
3. [Model Architecture (From Scratch)](#Model-Architecture-From-Scratch)
4. [Baseline Implementation](#Baseline-Implementation)
5. [Optimization Techniques](#Optimization-Techniques)
6. [Evaluation Metrics](#Evaluation-Metrics)
7. [Ablation Study](#Ablation-Study)
    - [Learning Rate Comparisons](#Learning-Rate-Comparisons)
    - [Regularization](#Regularization)
        - [Effect of Dropout Probability](#Effect-of-Dropout-Probability)
        - [Effect of L2 Regularization Strength](#Effect-of-L2-Regularization-Strength)
    - [Effect of Optimization Algorithm](#Effect-of-Optimization-Algorithm)
    - [Effect of Adam Hyperparameters](#Effect-of-Adam-Hyperparameters)
8. [Final Model Configuration](#Final-Model-Configuration)
9. [Results and Discussion](#Results-and-Discussion)
10. [Conclusion](#Conclusion)

## Project Overview

This project implements a neural network from scratch to classify CIFAR-10 images.  
The objective is to achieve at least 75% test accuracy.  

An ablation study is conducted to evaluate:
- Learning rate decay strategies  
- Regularization methods (L2, Dropout)  
- Optimization algorithms (SGD, Momentum, Adam)  
- Adam hyperparameter sensitivity  

## Dataset

In [ ]:
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from util import get_device
from evaluations import train_model
from model.model import Model
import os

In [ ]:
# CIFAR-10 normalization stats
cifar_mean = [0.485, 0.456, 0.406]
cifar_std  = [0.229, 0.224, 0.225]


# Training transforms (with augmentation + normalization)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean, std=cifar_std)
])


# Test transforms (NO augmentation, but WITH normalization)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean, std=cifar_std)
])


training_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=train_transform
)

test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=test_transform
)

#workers
workers = os.cpu_count()
print(f"Setting number of workers to available CPU cores: {workers}")
from torch.utils.data import DataLoader
batch_size = 128
train_loader = torch.utils.data.DataLoader(training_data, batch_size=batch_size, shuffle=True, num_workers=workers, pin_memory=True, persistent_workers=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True, persistent_workers=True)

### Data Format & Batch Creation

In [ ]:
from model.model import Model
image, label = training_data[0]
print(image.shape)
print("Label: ", label)

print("All Class Labels: \n", training_data.classes)

for images, labels in train_loader:
    print("images: [batch_size, channels, height, width]")
    print(images.shape)
    print("labels: [batch_size]")
    print(labels.shape)

    break


# Example images

In [ ]:
from util import show_random_images
show_random_images(training_data)


## Create model (Pytorch Implementation)

### Layer Creation

In [ ]:
device = get_device()

In [ ]:
main_model = Model()
train_model(
    model=main_model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    epochs=100,
    lr=0.001,
    optimizer_type="csi5140_adam",
    weight_decay=1e-4,
    learn_rate_type="cosine",
    gamma=0.9
)

In [ ]:
# torch.save(model.state_dict(), "cifar10_model_adam_normalize.pth")

# Abalation Study

## Base Model

In [ ]:
base_model, base_train_accs, base_test_accs, base_train_costs = train_model(
    model=Model(),
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    epochs=5,
    lr=0.01,
    optimizer_type="sgd",
    weight_decay=0.0,
    learn_rate_type=None
)
print(f"Base Model Stats: \nTrain Accuracy (Per Epoch): {base_train_accs}\nTest Accuracy (Per Epoch): {base_test_accs} \nLoss Per Iteration: {base_train_costs}")

## Regularization

### L2 Regularization 

### Dropout

## Optimization Algorithms

### Cosine Decay

### Adam - Beta1 & Beta2